# 1. Importação de Bibliotecas e Módulos

Nesta etapa inicial, importamos as bibliotecas fundamentais para a execução do projeto:
- **Pandas**: Para leitura e manipulação da base de dados.
- **Scikit-Learn**: Para divisão da base de dados (`train_test_split`), vetorização de texto (`TfidfVectorizer`), modelo de classificação (`LogisticRegression`), gerenciamento de fluxo (`Pipeline`) e métricas de avaliação de desempenho (`accuracy_score`, `classification_report`, `confusion_matrix`).

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 2. Carregamento dos Dados e Divisão Treino / Teste

Carregamos os dados a partir do arquivo Excel `dados/textos.xlsx`, garantindo que a coluna `resp_text` seja tratada explicitamente como string.

Em seguida, realizamos a separação entre variáveis independentes (`X`) e dependentes (`y`):
- **`X`**: Coluna de texto das respostas (`resp_text`).
- **`y`**: Rótulo da clareza do texto (`clarity`).

A base é dividida em **80% para treino** e **20% para teste**, utilizando **amostragem estratificada** (`stratify=df["clarity"]`) para manter a distribuição original das classes em ambos os conjuntos.

In [11]:
df = pd.read_excel("dados/textos.xlsx",dtype={"resp_text": str})

df.head()

X_train, X_test, y_train, y_test = train_test_split(
    df["resp_text"],
    df["clarity"],
    test_size=0.2,
    random_state=42,
    stratify=df["clarity"]
)

# 3. Validação dos Tipos de Dados em `resp_text`

Antes do treinamento e da vetorização dos textos, fazemos uma verificação de consistência dos tipos de dados na coluna `resp_text`.

Essa etapa garante que todas as entradas sejam efetivamente do tipo string (`<class 'str'>`), prevenindo eventuais falhas durante a extração de recursos (TF-IDF).

In [12]:
print(df["resp_text"].apply(type).value_counts())

resp_text
<class 'str'>    20092
Name: count, dtype: int64


# 4. Construção e Treinamento do Pipeline de Classificação

Definimos e treinamos um pipeline sequencial encadeando duas etapas principais:
1. **`TfidfVectorizer`**: Converte os textos em matrizes numéricas TF-IDF considerando unigramas e bigramas (`ngram_range=(1, 2)`) e limitando os termos mais frequentes em `10.000` (`max_features=10000`).
2. **`LogisticRegression`**: Modelo de Regressão Logística para classificação supervisionada com limite de iterações ajustado para convergência (`max_iter=1000`).

In [13]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ('logistic_regression', LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)

# 5. Avaliação do Modelo

Realizamos as predições no conjunto de teste (`X_test`) e avaliamos o desempenho do modelo utilizando o relatório de classificação (`classification_report`) e a acurácia global (`accuracy_score`).

In [ ]:
y_pred = pipeline.predict(X_test)

print("Acurácia:", accuracy_score(y_test, y_pred))
print("\nRelatório de Classificação:\n", classification_report(y_test, y_pred))